In [ ]:
%load_ext autoreload
%autoreload 2

import sys, importlib, shutil
from pathlib import Path

workspace_root = str(Path.cwd().parent)
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
    
shutil.rmtree(Path(workspace_root) / 'utils/__pycache__', ignore_errors=True)

import utils.experiment
importlib.reload(utils.experiment)

from utils.experiment import Experiment
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()
backend = service.backend("ibm_kingston")
print(f"Using backend: {backend.name}")
# backend = None  # Set to None to use the default simulator backend

a = 1  # decay constant
Z = 1  # nuclear charge
center_distance = 1.4
n_qubits = 5
max_range = 16
shots = 1024

exp = Experiment(
    allow_measurement=False,
    optimize_t_gates=True,
    enable_twirling=False,
    enable_measure_mitigation=False,
    enable_zne=True,
    ibm_backend=backend,
    zne_noise_factors=[1, 2, 3],
    zne_extrapolator = 'linear',
)
experiment_results = exp.run_single_s1_1d_overlap_integral(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_analytical(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_finite_diff(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_integral(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_s2_s1_3d_overlap_integral(n_qubits, a, max_range, shots)

# experiment_results = exp.run_s1_s2_3d_overlap_integral(n_qubits, a, max_range, shots)


experiment_results.print()




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using backend: ibm_kingston


c:\Users\sorin\anaconda3\envs\qiskit21\Lib\site-packages\qiskit_ibm_runtime\qiskit_runtime_service.py:1210: UserWarning: This instance has met its usage limit. Workloads will not run until time is made available. Check https://quantum.cloud.ibm.com/instances/crn%3Av1%3Abluemix%3Apublic%3Aquantum-computing%3Aus-east%3Aa%2F4db82d766fa147ed957fb4b3620ae0d6%3A6ae8f6bc-cd65-400d-aa95-b33f06ed5dd0%3A%3A for more details.
  warnings.warn(
c:\Users\sorin\anaconda3\envs\qiskit21\Lib\site-packages\qiskit_ibm_runtime\fake_provider\local_service.py:274: UserWarning: Options {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4'}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")
c:\Users\sorin\anaconda3\envs\qiskit21\Lib\site-packages\qiskit_ibm_runtime\fake_provider\local_service.py:274: UserWarning: Options {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4'}, 'resilience': {'zne_mitigation': True, 'zne':

KeyboardInterrupt: 

# 2p Orbital in Spherical Coordinates — MPS Analysis

## Background: how the 1s and 2s circuits encode the radial integral

In spherical coordinates the 3D inner product of two radial orbitals is

$$\langle A | B \rangle = \int_0^\infty \chi_A^*(r)\,\chi_B(r)\,dr$$

where $\chi(r) = r \cdot R(r)$ is the *reduced* radial function and $R(r)$ is the ordinary radial part.
Including the $r$ factor in the amplitude lets us evaluate the 3D overlap as a plain 1D sum:

$$\langle A | B \rangle \approx \sum_{r=0}^{2^n-1} f_A(r)^*\, f_B(r)$$

The MPS circuits encode $f(r)$ as quantum amplitudes.  The "Jacobian" versions therefore multiply by the extra factor of $r$:

| Orbital | Amplitudes $f(r)$ | Bond dim $\chi$ |
|---|---|---|
| 1s (Jacobian) | $r \cdot e^{-\alpha r}$ | 2 |
| 2s (Jacobian) | $r \cdot (2 - \alpha r)\, e^{-\alpha r/2}$ | 3 |
| **2p (Jacobian)** | $r^2 \cdot e^{-\alpha r/2}$ | **3** |

---

## Transfer matrix structure for 2p

The radial coordinate is represented in binary: $r = \sum_k s_k\, p_k$ with $p_k = 2^{n-1-k}$.

The bond tracks the tuple $(1,\; r_\text{acc},\; r_\text{acc}^2)$ so that after processing all $n$ bits the right boundary can extract $r^2$.

The interior transfer matrices are **identical** to those of the 2s Jacobian:

$$A^{[k]0} = I_3, \qquad A^{[k]1} = e_k \begin{pmatrix} 1 & p_k & p_k^2 \\ 0 & 1 & 2p_k \\ 0 & 0 & 1 \end{pmatrix}, \qquad e_k = e^{-\alpha p_k/2}$$

When bit $s_k = 1$ is added the accumulator updates as:
- $r_\text{new} = r_\text{acc} + p_k$, so $v_1 \to v_1 + p_k$
- $r_\text{new}^2 = r_\text{acc}^2 + 2p_k r_\text{acc} + p_k^2$, so $v_2 \to v_2 + 2p_k v_1 + p_k^2$

This matches the matrix exactly.

### Boundary vectors

The only difference from 2s Jacobian is the **right boundary**, which selects which polynomial of $r$ to extract:

| | Left boundary | Right boundary |
|---|---|---|
| 2s Jacobian | $(1,\; 0,\; 0)$ | $(0,\; 2,\; -\alpha)^T$ — extracts $r(2-\alpha r)$ |
| **2p Jacobian** | $(1,\; 0,\; 0)$ | $(0,\; 0,\; 1)^T$ — extracts $r^2$ |

### Concrete MPS tensors

**k = 0** (left boundary, shape $1\times 2\times 3$):
$$A^{[0]0} = \tfrac{1}{N}(1,\;0,\;0), \qquad A^{[0]1} = \tfrac{e_0}{N}(1,\;p_0,\;p_0^2)$$

**0 < k < n-1** (interior, shape $3\times 2\times 3$):
$$A^{[k]0} = I_3, \qquad A^{[k]1} = e_k\begin{pmatrix}1&p_k&p_k^2\\0&1&2p_k\\0&0&1\end{pmatrix}$$

**k = n-1** (right boundary, shape $3\times 2\times 1$):
$$A^{[n-1]0} = (0,\;0,\;1)^T, \qquad A^{[n-1]1} = (e_k p_k^2,\;2 e_k p_k,\;e_k)^T$$

---

## Norm

$$\|f\|^2 = \sum_{r=0}^{2^n-1} r^4\, e^{-\alpha r}$$

Computed via the same $9\times 9$ transfer matrix product as the 2s Jacobian norm, but with right boundary $(0,0,1)$ instead of $(0,2,-\alpha)$:

```python
l = np.array([1.0, 0.0, 0.0])
r_vec = np.array([0.0, 0.0, 1.0])   # ← only change from 2s norm
v = np.kron(l, l)
for k in range(n):
    p_k = 2.0 ** (n - 1 - k)
    e_k = np.exp(-p_k * alpha / 2.0)
    N_k = e_k * np.array([[1, p_k, p_k**2], [0, 1, 2*p_k], [0, 0, 1]])
    T_k = np.eye(9) + np.kron(N_k, N_k)
    v = v @ T_k
norm_sq = v @ np.kron(r_vec, r_vec)
```

---

## Angular part

The 1s and 2s orbitals have $l=0$, so their spherical harmonic $Y_0^0$ is a constant and the angular integral always equals 1 — the circuit only needs to encode the radial part.

The 2p orbital has $l=1$ with spherical harmonics $Y_1^m$.  Two consequences:

- **Self-overlap** $\langle 2p | 2p \rangle$: the angular integral $\int |Y_1^m|^2\,d\Omega = 1$, so only the radial MPS above is needed.
- **Cross-overlaps** $\langle 1s | 2p \rangle$ and $\langle 2s | 2p \rangle$: the angular integral $\int Y_0^0 (Y_1^m)^*\,d\Omega = 0$ by orthogonality, so these are exactly zero regardless of the radial part — no circuit is needed.
- **Matrix elements with an anisotropic potential** (e.g. $V \propto \cos\theta$): the angular integral no longer factorises to 1 or 0, and you would need a separate angular register encoding $\cos\theta$. For the isotropic Coulomb potential the angular part still factorises and can be computed analytically.

Hydrogen 2p vs STO-2p
For the 2p they are the same functional form — unlike 2s, the STO is exact for hydrogen 2p.

Exact hydrogen 2p (atomic units, Z=1)
$$\psi_{2p_z}(r,\theta,\phi) = \frac{1}{4\sqrt{2\pi}}, r, e^{-r/2}, \cos\theta$$

Radial part only: $R_{21}(r) \propto r, e^{-r/2}$

STO-2p definition
By the Slater-type orbital convention for quantum numbers $(n=2, l=1)$:

$$\psi_{2p}^{\text{STO}}(r,\theta,\phi) = N, r^{n-1}, e^{-\zeta r}, Y_1^m(\theta,\phi) = N, r, e^{-\zeta r}, Y_1^m$$

With $\zeta = 1/2$ (the Slater rule for the second shell), this exactly reproduces the hydrogen 2p. No approximation.

Why 2s is different
Orbital	Exact hydrogen radial	STO radial	Same?
1s	$e^{-r}$	$e^{-\zeta r}$	✓ exact
2p	$r, e^{-r/2}$	$r, e^{-\zeta r}$	✓ exact
2s	$(1 - r/2), e^{-r/2}$	$r, e^{-\zeta r}$	✗ no node
The hydrogen 2s has a radial node at $r = 2a_0$ from the polynomial factor $(1-r/2)$. The bare STO $r, e^{-\zeta r}$ has no node. That is why the code uses $(2-\alpha r), e^{-\alpha r/2}$ — the polynomial is there to build in the node and maintain orthogonality with 1s.

Consequence for the MPS
The MPS you would implement for 2p (Jacobian encoding $f(r) = r^2 e^{-\alpha r/2}$) is not an approximation — it encodes the exact hydrogen 2p radial function. No bond dimension growth or polynomial correction is needed beyond what the $r^2$ factor already requires (χ = 3).



Why 1s and 2s are radially orthogonal
Both have $l=0$, so the full overlap is purely the radial integral (angular part $\int|Y_0^0|^2 d\Omega = 1$):

$$\langle f_{1s} | f_{2s} \rangle \propto \int_0^\infty r(2-\alpha r), e^{-\alpha r/2} \cdot r, e^{-\alpha r} , dr = 2\cdot\frac{6}{\alpha^4} - \alpha\cdot\frac{12}{\alpha^5} = \frac{12}{\alpha^4} - \frac{12}{\alpha^4} = 0$$

Exactly zero — that's why the compute/uncompute circuit works for them.

Why 2s and 2p are NOT radially orthogonal
$$\langle f_{2s} | f_{2p} \rangle \propto \int_0^\infty r(2-\alpha r)e^{-\alpha r/2} \cdot r^2 e^{-\alpha r/2}, dr = \int_0^\infty r^3(2-\alpha r)e^{-\alpha r},dr = 2\cdot\frac{6}{\alpha^4} - \alpha\cdot\frac{24}{\alpha^5} = -\frac{12}{\alpha^4} \neq 0$$

They are orthogonal in 3D only because of the angular part: $\int Y_0^{0*}, Y_1^m, d\Omega = 0$. The radial MPS circuits know nothing about that.

The angular integral over the full sphere with the correct measure $d\Omega = \sin\theta, d\theta, d\phi$ is:

$$\int Y_0^{0*}, Y_1^0, d\Omega = \int_0^{2\pi} d\phi \int_0^\pi \frac{1}{\sqrt{4\pi}} \cdot \sqrt{\frac{3}{4\pi}}\cos\theta \cdot \sin\theta, d\theta = \frac{\sqrt{3}}{2} \int_0^\pi \cos\theta\sin\theta, d\theta$$

The key is the $\sin\theta$ in the measure. Substituting $u = \cos\theta$, $du = -\sin\theta, d\theta$:

$$\int_0^\pi \cos\theta\sin\theta, d\theta = \int_1^{-1} u,(-du) = \int_{-1}^{1} u, du = \left[\frac{u^2}{2}\right]_{-1}^{1} = \frac{1}{2} - \frac{1}{2} = 0$$

Geometrically: $\cos\theta$ is positive in the northern hemisphere and negative in the southern hemisphere by exactly the same amount. The $\sin\theta$ measure is symmetric about the equator. So the two halves cancel exactly.

This is why $Y_0^0$ (constant) and $Y_1^0$ ($\propto \cos\theta$) are orthogonal — it is simply the statement that $\cos\theta$ integrates to zero over the full sphere, regardless of the constant it is multiplied by.